# Who Pays to Know?

## Computational Baseline for Strategic Information Acquisition

COMSCI/ECON 206 PS1 v2. This notebook runs a formal two-agent game and a discrete cost sweep. It reports equilibrium calculations, not observations of AI or human behavior.

## 1. Research question

When information is costly but collectively valuable, when will agents acquire it themselves rather than free-ride on another agent? This notebook tests the author's initial pure-equilibrium prediction with reproducible computation.

## 2. Model

Agents A and B each choose **Research (R)** or **Skip (S)**. If at least one researches, both receive information value **V**; each researcher privately pays cost **c**. If both skip, both receive zero. One research action suffices to produce the full information benefit in this simplified model. Rows are A's actions and columns are B's, ordered **[Research, Skip]**.

## 3. Baseline prediction before computation

Before the computational analysis, the author predicted two **pure** Nash equilibria: **(Research, Skip)** and **(Skip, Research)**. At this point in the reasoning these are theoretical predictions, not computational results. The cells below test them.

In [1]:
from pathlib import Path
import sys
import subprocess

# Colab opens a single notebook file, so fetch the public repository that holds
# the verified model. Local Jupyter uses the neighboring repository directly.
try:
    import google.colab  # noqa: F401
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    repo_url = "https://github.com/micL1222/PS1-Yiqiao.git"
    repo_branch = "v2-information-acquisition"
    project_root = Path("/content/ps1-v2-information-acquisition")
    source_file = project_root / "src" / "information_acquisition_game.py"
    if not source_file.is_file():
        if project_root.exists():
            raise RuntimeError("The Colab repository directory exists without the expected verified model")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--single-branch", "--branch", repo_branch,
             repo_url, str(project_root)],
            check=True,
        )
    else:
        actual_remote = subprocess.run(
            ["git", "-C", str(project_root), "remote", "get-url", "origin"],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        if actual_remote != repo_url:
            raise RuntimeError("The existing Colab checkout is from a different repository")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "numpy>=1.24,<3",
         "pandas>=2,<3", "nashpy==0.0.41"],
        check=True,
    )
else:
    candidates = (Path.cwd(), Path.cwd().parent)
    project_root = next(
        (candidate for candidate in candidates
         if (candidate / "src" / "information_acquisition_game.py").is_file()),
        None,
    )
    if project_root is None:
        raise RuntimeError("Run this notebook from the repository root or notebooks/ directory")

import pandas as pd
import numpy as np
sys.path.insert(0, str(project_root))
from src.information_acquisition_game import (
    ACTIONS, payoff_matrices, pure_nash_equilibria,
    support_enumeration_results, interior_mixed_research_probability,
)
from src.run_baseline_analysis import analyze_case
print("Using source implementation from:", (project_root / "src" / "information_acquisition_game.py").relative_to(project_root))
print("Action order:", ACTIONS)


Using source implementation from: src/information_acquisition_game.py
Action order: ('Research', 'Skip')


## 4. Construct the V = 4, c = 2 baseline game

In [2]:
V, c = 4, 2
agent_a_payoffs, agent_b_payoffs = payoff_matrices(V, c)
print("Agent A payoff matrix (rows A, columns B):")
print(agent_a_payoffs)
print("Agent B payoff matrix (rows A, columns B):")
print(agent_b_payoffs)
assert np.array_equal(agent_a_payoffs, [[2, 2], [4, 0]])
assert np.array_equal(agent_b_payoffs, [[2, 4], [2, 0]])

Agent A payoff matrix (rows A, columns B):
[[2. 2.]
 [4. 0.]]
Agent B payoff matrix (rows A, columns B):
[[2. 4.]
 [2. 0.]]


## 5. Verify pure equilibria independently

The reusable best-response checker evaluates all four action profiles. Ties count as best responses.

In [3]:
pure = pure_nash_equilibria(V, c)
print("Independent pure Nash equilibria:", pure)
predicted = {("Research", "Skip"), ("Skip", "Research")}
assert set(pure) == predicted
print("Initial pure-equilibrium prediction: PASS")

Independent pure Nash equilibria: [('Research', 'Skip'), ('Skip', 'Research')]
Initial pure-equilibrium prediction: PASS


## 6. NashPy support enumeration

Each probability vector follows [Research, Skip]. The classification is based on the returned probabilities.

In [4]:
nashpy_result = support_enumeration_results(V, c)
for item in nashpy_result["equilibria"]:
    print(item["classification"], "A =", item["agent_a"], "B =", item["agent_b"])
nashpy_pure = {
    tuple(item["pure_profile"]) for item in nashpy_result["equilibria"]
    if item["classification"] == "pure"
}
assert nashpy_pure == set(pure)
print("NashPy and independent pure-equilibrium sets agree: PASS")
print("NashPy warnings:", nashpy_result["warnings"])

pure A = [1.0, 0.0] B = [0.0, 1.0]
pure A = [0.0, 1.0] B = [1.0, 0.0]
mixed A = [0.5, 0.5] B = [0.5, 0.5]
NashPy and independent pure-equilibrium sets agree: PASS
NashPy warnings: []


## 7. Mixed-equilibrium cross-check

For the strict interior case **0 < c < V**, Research gives V − c, while Skip gives pV when the other agent researches with probability p. Making these equal yields **p(Research) = (V − c)/V = 1 − c/V**. This calculation was applied after the initial two-pure-equilibrium prediction and does not characterize degenerate boundaries.

In [5]:
analytical_p = interior_mixed_research_probability(V, c)
mixed = [item for item in nashpy_result["equilibria"] if item["classification"] == "mixed"]
assert len(mixed) == 1
target = [analytical_p, 1 - analytical_p]
assert np.allclose(mixed[0]["agent_a"], target, atol=1e-9, rtol=0)
assert np.allclose(mixed[0]["agent_b"], target, atol=1e-9, rtol=0)
print("Analytical p(Research):", analytical_p)
print("NashPy mixed probabilities: A =", mixed[0]["agent_a"], "; B =", mixed[0]["agent_b"])
print("Analytical/NashPy cross-check: PASS")

Analytical p(Research): 0.5
NashPy mixed probabilities: A = [0.5, 0.5] ; B = [0.5, 0.5]
Analytical/NashPy cross-check: PASS


## 8. Meaningful modification: vary research cost

Hold V = 4 and compute c = 0, 1, 2, 3, 4, 5. Boundary cases c = 0 and c = V are degenerate; the interior mixed formula is not applied there. The finite NashPy output at a boundary does not claim to exhaust a possible continuum of equilibria.

In [6]:
cases = [analyze_case(V, current_cost) for current_cost in range(6)]
assert all(case["pure_methods_agree"] for case in cases)
assert all(
    case["analytical_and_nashpy_mixed_agree"]
    for case in cases if case["interior_analytical_mixed_applies"]
)
summary_rows = []
for case in cases:
    mixed_p = case["nashpy_mixed_p_research"]
    summary_rows.append({
        "c": case["parameters"]["c"],
        "case": case["relation_to_V"],
        "pure equilibria": ", ".join(str(tuple(p)) for p in case["independent_pure_equilibria"]),
        "number pure": case["number_of_pure_equilibria"],
        "analytical p(R)": case["analytical_p_research"],
        "NashPy mixed p(R), A": None if mixed_p is None else mixed_p["agent_a"],
        "NashPy mixed p(R), B": None if mixed_p is None else mixed_p["agent_b"],
        "mixed cross-check": case["analytical_and_nashpy_mixed_agree"],
    })
display(pd.DataFrame(summary_rows))

,c,case,pure equilibria,number pure,analytical p(R),"NashPy mixed p(R), A","NashPy mixed p(R), B",mixed cross-check
0,0,zero-cost boundary,"('Research', 'Research'), ('Research', 'Skip')...",3,NaN,NaN,NaN,None
1,1,interior,"('Research', 'Skip'), ('Skip', 'Research')",2,0.75,0.75,0.75,True
2,2,interior,"('Research', 'Skip'), ('Skip', 'Research')",2,0.50,0.50,0.50,True
3,3,interior,"('Research', 'Skip'), ('Skip', 'Research')",2,0.25,0.25,0.25,True
4,4,equal-value boundary,"('Research', 'Skip'), ('Skip', 'Research'), ('...",3,NaN,NaN,NaN,None
5,5,high-cost,"('Skip', 'Skip')",1,NaN,NaN,NaN,None


## 9. Interpretation

The executed baseline cells test the author's two predicted pure equilibria against independent best responses and NashPy. NashPy also supplies a mixed profile, which the interior analytical formula cross-checks. The sweep shows how the computed pure profiles and interior mixing probability change across these six costs. Direct enumeration covers all four **pure** profiles at each cost; the finite support-enumeration output should not be taken as a complete description of degenerate boundary correspondences.

## 10. Evidence limits

These are equilibrium calculations for a specified formal game. They are not synthetic agent-behavior results and provide no observed evidence about real LLM or human choices. They do not establish educational effectiveness of a future Hugging Face demo.